In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Fig. 4: how the early discs lose their rotation, and why

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/letter/fig4_scrambling.ipynb)

Every number in the text and caption is written by `scripts/paper_text_letter.py` from the result tables
(`results/macros_letter.tex`); the code cells are the figure script itself (`scripts/paper_letter_scramble.py`), step by step.

**What it shows.** Figure 4 follows the earliest stars of the progenitors of simulated analogues of the Large
Magellanic Cloud (LMC) in TNG50 ([Pillepich et al. 2019](https://doi.org/10.1093/mnras/stz2338)) and asks how fast they lose their
rotation, what removes it, and why only early. It uses 92 cohorts born at redshifts $z$ =
4–1.5 in 16 of the 20 progenitors with outputs at
every snapshot, and the dark matter of a larger sample of 60 progenitors that includes them. A cohort is the
stars formed in the 25 Myr before a simulation output (its birth snapshot), followed for 0.5
Gyr. Its rotational support is $v_\phi/\sigma_z$, the mean azimuthal velocity over the vertical velocity dispersion, measured about
the cohort's spin at birth. The same stars are also moved as test particles through their progenitor's gravity, computed with
AGAMA ([Vasiliev 2019](https://doi.org/10.1093/mnras/sty2672)): frozen at birth, with only the dark matter aspherical, and made
axisymmetric about the cohort's spin. Time is also given in orbital periods $T_{\rm orb}$ at 2 kpc
($T_{\rm orb}$ = 0.2 Gyr).

The shape of the dark-matter halo is measured by $\epsilon_2$: the amplitude of the quadrupole of the dark-matter potential at
$r$ = 2 kpc that is not axisymmetric about the disc's spin, relative to the monopole. It grows with a tilt of
the halo relative to the disc and with the halo's elongation in the disc plane, but not with a flattening along the spin. The
misalignment $\psi$ is the angle between the minor axis of the dark matter within 3 kpc and the spin of the
stellar disc.

(a) The rotation of the early cohorts is halved within one orbital period. The test particles keep their rotation only when the
gravity is made axisymmetric about their spin, so the loss is caused by the non-axisymmetric gravity of the young halo. (b) The
rotation that survives falls with $\epsilon_2$ at birth. (c) Followed in lookback time (the time before the present), the halo
settles, lining up with the disc and becoming rounder, only after the LMC's oldest stars formed.

**Starts from:** the quick route.

## Making the figure, step by step

<details><summary>About <code>scripts/paper_letter_scramble.py</code></summary>

```text
Letter figure: how fast the early simulated cohorts lose their rotation, what does it, and why only early.

Statistic: per galaxy the median in each time bin, then the median over galaxies; the shading
spans the 16th-84th percentiles, darkest at the median and fading towards both percentiles (density_shade).
(a) How fast and what does it: TNG50 cohorts born at z = 1.5-4 (results/sims_scramble.csv, scripts/sims_scramble.py: the 25 Myr of
    stars before a birth snapshot, 92 cohorts in 16 progenitors; as Fig. sims_scramble a): v_phi/sigma_z about the cohort's spin at
    its birth snapshot, held fixed (vs_meas_birth, vs_sim_birth), of the members at r < 3 kpc and cylindrical R > 0.5 kpc at that
    snapshot, followed wherever they go, against time since the birth snapshot in bins 0 / 0.05-0.2 / 0.2-0.35 / 0.35-0.5 Gyr: the
    real stars, and the same stars re-run as test particles in parts of the gravity: all gravity frozen at birth (E); only the dark
    matter non-round, the rest made axisymmetric (G_DM); everything axisymmetric about the cohort's spin (D0). (At formation, from
    BirthVel, the same galaxies' stars of these epochs have 2.05, 16-84 per cent 1.48-2.58: Fig. sims_birth.) The TNG50
    star-forming gas (results/sims_heating_gas.csv, scripts/build_sims_heating.py: 60 progenitors, 0.5 < R < 3 kpc, about its own
    spin; blue) at z >= Z_GAS, per galaxy the median over those snapshots; shading as for the stars, over these 60 values. Dotted: the LMC's [Fe/H] < -1.5 giants (results/kin_support_ellipsoid.csv, ratio_z). Inner tick labels on the time axis: t / T_orb, T_orb the
    median orbital period at 2 kpc of the progenitors at the early cohorts' births (results/sims_vcirc.csv, scripts/
    build_sims_vcirc.py).
(b) What sets the loss: the real stars of (a), v_phi/sigma_z (fixed birth axis) in T_AFTER Gyr after the birth snapshot (per cohort the
    median), against the dark-matter quadrupole eps2 = |Phi_2| / |Phi_0| at 2 kpc of the cohort's galaxy at its birth snapshot
    (results/sims_halo_settle.csv, scripts/sims_halo_settle.py: AGAMA multipole of the dark matter in the disc frame; the l = 2 terms
    non-axisymmetric about the disc, |m| = 1 tilt and |m| = 2 in-plane elongation); quartiles of eps2, median and 16th-84th
    percentiles over cohorts. Dashed: the cohorts' median at the birth snapshot; dotted: the LMC.
(c) How the halo evolves: against lookback time (0.5-Gyr bins; per galaxy the median, then the median over the 60 progenitors of
    results/sims_halo_settle.csv): the angle psi between the halo's minor axis (dark matter within 3 kpc) and the stellar spin (left
    axis, 16th-84th percentiles shaded), and on the right axis eps2 and its part from the halo's elongation in the disc plane
    (eps_elong; the other part, the tilt of the flattening to the disc, eps_tilt, follows psi and is printed for the text).
Writes figures/paper_letter_scramble.png.
Run: .venv/bin/python scripts/manifest.py run paper_letter_scramble.py
```
</details>

**Data read by `scripts/paper_letter_scramble.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/sims_halo_settle.csv` | repository | written by `python scripts/sims_halo_settle.py` |
| `results/sims_heating_gas.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_scramble.csv` | repository | written by `python scripts/sims_scramble.py` |
| `results/sims_vcirc.csv` | repository | written by `python scripts/build_sims_vcirc.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

### Settings and helpers
The time bins, and two helpers: `gal_median` takes the median per galaxy and then over galaxies, and `density_shade` draws the
spread between the 16th and 84th percentiles, darkest at the median.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, to_rgba

from lmcold.layout import panel_grid
from lmcold.sources import ROOT

T_AFTER = (0.2, 0.35)       # Gyr after the birth snapshot: v_phi/sigma_z in (b), about 1-1.5 orbits
TIME_EDGES = (0.0, 0.05, 0.2, 0.35, 0.5)   # Gyr since the birth snapshot: bins of (a)
N_EPS = 4                  # (b): cohorts in quantiles of eps2
EPOCH = 0.5                # Gyr, lookback-time bins of (c)
Z_GAS = 2.0                # star-forming gas in (a) at z >= Z_GAS (lookback > 10.4 Gyr), when the oldest cohorts formed
RES = ROOT / 'results'
GAS_COLOUR = '#2b6cb0'      # blue of the gas shading in (a) (PI); the birth colour of Fig. sims_birth
LEG = dict(fontsize=5, frameon=False, handlelength=1.5, columnspacing=0.8)   # legends in the top right corners (PI)


def gal_median(d, x, y):
    """Per-galaxy median of y in each value of column x (the plotted time), then the median and 16th/84th percentiles over galaxies.

    d: table with columns x, y and halo (one row per cohort, variant and time); returns a DataFrame with columns t (the values of x),
    m (median), lo, hi (16th, 84th percentiles over the galaxies), one row per value of x."""
    g = d.groupby([x, 'halo'])[y].median().groupby(level=0)
    return pd.DataFrame({'m': g.median(), 'lo': g.quantile(.16), 'hi': g.quantile(.84)}).rename_axis('t').reset_index()


def density_shade(a, x, samples, colour, peak, ylim):
    """Shade the spread of the values in samples[i] (one array per x[i], x increasing) between their 16th and 84th percentiles,
    darkest at the median and fading towards both percentiles: opacity peak * (1 - |2F - 1| / 0.68), F the fraction of values below
    (from the percentiles, linear between them), with the percentiles linear in x (in log x on a log axis) between the nodes, on a
    grid over ylim. (A kernel density of 16 values is nearly flat inside this range: its smoothing width is about the range itself.)

    a: axes to draw on; x: the n positions on the horizontal axis; samples: n arrays of values (axis units of y); colour: matplotlib
    colour; peak: largest opacity (0-1); ylim: (bottom, top) of the shaded grid, in the units of y."""
    q = np.linspace(.16, .84, 35)
    Q = np.array([np.quantile(v, q) for v in samples])
    f = np.log if a.get_xscale() == 'log' else (lambda z: z)
    xs = np.linspace(f(x[0]), f(x[-1]), 400)
    y = np.linspace(*ylim, 600)
    img = np.zeros((len(y), len(xs)))
    for j, Qx in enumerate(np.array([np.interp(xs, f(np.asarray(x)), Q[:, i]) for i in range(len(q))]).T):
        F = np.interp(y, Qx, q, left=np.nan, right=np.nan)
        img[:, j] = np.nan_to_num(1 - np.abs(2 * F - 1) / 0.68)
    cmap = LinearSegmentedColormap.from_list('', [to_rgba(colour, 0), to_rgba(colour, peak)])
    a.pcolormesh(np.exp(xs) if f is np.log else xs, y, img, cmap=cmap, vmin=0, vmax=1, shading='nearest', lw=0, antialiased=False, zorder=0)

### Load the data
The figure reads five result tables and no simulation; the TNG50 progenitors are needed only to remake the four simulation tables:
`results/sims_scramble.csv` (`scripts/sims_scramble.py`: the early cohorts' real stars and their test-particle re-runs),
`results/sims_heating_gas.csv` (`scripts/build_sims_heating.py`: the star-forming gas of 60 progenitors),
`results/sims_halo_settle.csv` (`scripts/sims_halo_settle.py`: shape and quadrupole of the dark matter of the same 60 progenitors)
and `results/sims_vcirc.csv` (`scripts/build_sims_vcirc.py`: their orbital period at 2 kpc). The LMC value is the [Fe/H] < -1.5
giants' $v_{\rm rot}/\sigma_z$ from the stage-1 ellipsoid fits, `results/kin_support_ellipsoid.csv`.

In [ ]:
sc = pd.read_csv(RES / 'sims_scramble.csv', dtype={'halo': str}).replace([np.inf, -np.inf], np.nan)   # v_phi/sigma_z per cohort, variant, time; inf: 1-2 particles left
gas = pd.read_csv(RES / 'sims_heating_gas.csv', dtype={'halo': str}).query('z >= @Z_GAS')   # star-forming gas per progenitor and snapshot, z >= Z_GAS
T = pd.read_csv(RES / 'sims_halo_settle.csv', dtype={'halo': str})   # dark-matter misalignment, axis ratios, eps2 per progenitor and snapshot
t_orb = pd.read_csv(RES / 'sims_vcirc.csv').t_orb.median()   # median orbital period at 2 kpc at the early cohorts' births [Gyr]
lmc = pd.read_csv(RES / 'kin_support_ellipsoid.csv').set_index('population').ratio_z['giants [Fe/H] < -1.5']   # LMC v_rot/sigma_z

### (a) How fast the rotation is lost, and what does it
$v_\phi/\sigma_z$ of the early TNG50 cohorts against time since birth (`sc`, from `results/sims_scramble.csv`):
the real stars, and the same stars moved as test particles in the gravity frozen at birth, with only the dark matter aspherical,
and made axisymmetric about the cohort's spin. Also the star-forming gas (`gas`) and the LMC's [Fe/H] < -1.5 giants (`lmc`).
The orbital period at 2 kpc (`t_orb`) labels the time axis.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, ax = panel_grid(1, 3, width=7.0, left=0.45, right=0.5, hgap=0.55, top=0.3)
ax = ax[0]

# (a) how fast and what does it: v_phi/sigma_z of the early cohorts about their own spin, real stars and test particles in parts of
# the gravity; the star-forming gas they formed from and the LMC's oldest giants
sc = pd.concat([sc, sc[sc.variant == 'A'].assign(variant='real', vs_sim_birth=lambda d: d.vs_meas_birth)]).rename(columns={'t': 'dt'})
sc['tb'] = sc.groupby(pd.cut(sc.dt, np.r_[-1e-9, TIME_EDGES[1:-1], TIME_EDGES[-1] + 1e-6])).dt.transform('median')   # plotted at the bins' median time
# variant of results/sims_scramble.csv: legend label, colour, line width ('real': the real stars, vs_meas_birth)
VAR = {'real': ('stars', 'k', 1.6), 'E': ('frozen gravity', 'C1', 1.0), 'G_DM': ('DM aspherical only', 'C3', 1.0),
       'D0': ('axisymmetric', 'C4', 1.0)}
a = ax[0]
g = (gas.vphi / gas.sz).groupby(gas.halo).median()   # star-forming gas v_phi/sigma_z, per galaxy the median over its snapshots
density_shade(a, (-0.03, 0.46), (g, g), GAS_COLOUR, 0.6, (0, 3.5))
a.fill_between([], [], color=GAS_COLOUR, alpha=0.45, lw=0, label='gas')   # legend entry for the shading
print(f'(a) TNG50 star-forming gas at z >= {Z_GAS}: {g.median():.2f} ({g.quantile(.16):.2f}-{g.quantile(.84):.2f}), {len(g)} galaxies')
for v, (lab, col, lw) in VAR.items():
    s = gal_median(sc[sc.variant == v], 'tb', 'vs_sim_birth')
    if v == 'real':
        per = sc[sc.variant == v].groupby(['tb', 'halo']).vs_sim_birth.median().dropna()
        density_shade(a, s.t.values, [per[t].values for t in s.t], 'k', 0.35, (0, 3.5))
    a.plot(s.t, s.m, '-o', color=col, ms=2.5, lw=lw, label=lab)
    print(f'(a) {v}: ' + '; '.join(f'{r.t:.2f} Gyr: {r.m:.2f}' for r in s.itertuples()))
a.axhline(lmc, color='k', lw=0.8, ls=':')
a.text(0.45, lmc - 0.05, r'LMC, [Fe/H] $<-1.5$', ha='right', va='top', fontsize=6)
a.set(xlabel='time since birth [Gyr]', ylabel=r'$v_\phi/\sigma_z$', xlim=(-0.03, 0.46), ylim=(-0.5, 3.5))   # below 0: room for the t/T_orb labels
a.tick_params(axis='x', which='both', direction='out', top=False)   # time since birth: ticks out, below the axis
orb = a.secondary_xaxis('bottom', functions=(lambda x: x / t_orb, lambda x: x * t_orb))   # t/T_orb: ticks in, labels inside
orb.tick_params(axis='x', which='both', direction='in', pad=-11, labelsize=6)
orb.tick_params(axis='x', which='minor', length=0)
orb.spines['bottom'].set_visible(False)
orb.set_xticks([0.5, 1, 1.5, 2])
orb.set_xticks([0.02 / t_orb], [r'$t/T_{\rm orb}$'], minor=True)   # the scale's name where its 0 would be; T_orb printed for the caption
print(f'T_orb {t_orb:.3f} Gyr, LMC v/sigma {lmc:.2f}')

### (b) What sets the loss
The $v_\phi/\sigma_z$ that each cohort keeps 0.2-0.35 Gyr after birth, against the dark-matter quadrupole $\epsilon_2$ at 2 kpc of its
galaxy at its birth (`T`, from `results/sims_halo_settle.csv`), in quartiles of $\epsilon_2$.

In [ ]:
# (b) what sets the loss: v_phi/sigma_z of the same cohorts after T_AFTER Gyr against the halo's quadrupole at their birth snapshot
real = sc[sc.variant == 'real']
after = real[(real.dt > T_AFTER[0]) & (real.dt <= T_AFTER[1])].groupby(['halo', 's_birth']).vs_sim_birth.median().rename('vs')
k = after.reset_index().merge(T, left_on=['halo', 's_birth'], right_on=['halo', 'snap'], how='inner')
g = k.groupby(pd.qcut(k.eps2, N_EPS), observed=True)
# per quartile of eps2: its median eps2 (x), the median and 16th/84th percentiles of v_phi/sigma_z after T_AFTER, and the cohorts
q = pd.DataFrame({'x': g.eps2.median(), 'm': g.vs.median(), 'lo': g.vs.quantile(.16), 'hi': g.vs.quantile(.84), 'n': g.size()})
a = ax[1]
a.set_xscale('log')
density_shade(a, q.x.values, [x.vs.values for _, x in g], 'k', 0.35, (-0.3, 3.5))
a.plot(q.x, q.m, 'k-o', lw=1.6, ms=2.5, label=f'stars after {T_AFTER[0]:.1f}-{T_AFTER[1]:.2f} Gyr')
v0 = gal_median(real, 'tb', 'vs_sim_birth').m.iloc[0]
a.axhline(v0, color='k', lw=0.8, ls='--', label='at birth')
a.axhline(lmc, color='k', lw=0.8, ls=':', label=r'LMC, [Fe/H] $<-1.5$')
a.set(xscale='log', xlim=(0.003, 0.025), ylim=(-0.5, 3.5), xlabel=r'DM quadrupole $\epsilon_2$ at 2 kpc, at birth', ylabel=r'$v_\phi/\sigma_z$')
a.set_xticks([0.003, 0.005, 0.01, 0.02], ['0.003', '0.005', '0.01', '0.02'])
a.minorticks_off()
print(f'(b) {len(k)} cohorts; Spearman of v/sigma after with eps2 {k.vs.corr(k.eps2, method="spearman"):.2f}, eps_tilt '
      f'{k.vs.corr(k.eps_tilt, method="spearman"):.2f}, psi {k.vs.corr(k.misalign, method="spearman"):.2f}, lookback '
      f'{k.vs.corr(k.lookback, method="spearman"):.2f}; quartiles: ' + '; '.join(f'{r.x:.4f}: {r.m:.2f} ({r.lo:.2f}-{r.hi:.2f}) n {r.n}' for r in q.itertuples()))
a.legend(loc='upper right', **LEG)

### (c) How the halo settles
Against lookback time: the angle $\psi$ between the minor axis of the dark matter within 3 kpc and the stellar spin, and
$\epsilon_2$ with its part from the halo's elongation in the disc plane. The figure is saved at the end of this cell.

In [ ]:
def by_epoch(d, y):
    """Per-galaxy median of y in 0.5-Gyr bins of lookback time, then the median and 16th/84th percentiles over galaxies."""
    g = d.assign(tb=(d.lookback / EPOCH).round() * EPOCH).groupby(['tb', 'halo'])[y].median().groupby(level=0)
    return pd.DataFrame({'m': g.median(), 'lo': g.quantile(.16), 'hi': g.quantile(.84)})


# (c) how the halo evolves: misalignment and quadrupole against time
a = ax[2]
psi = by_epoch(T, 'misalign')   # deg: angle between the halo's minor axis and the stellar spin, per 0.5-Gyr bin of lookback time
tb = (T.lookback / EPOCH).round() * EPOCH
per = T.groupby([tb, 'halo']).misalign.median()
density_shade(a, psi.index.values, [per[t].values for t in psi.index], 'C3', 0.35, (0, 80))
a.plot(psi.index, psi.m, '-o', color='C3', lw=1.4, ms=2.5, label=r'misalignment $\psi$')
a.set(xlim=(12.7, 7.3), ylim=(0, 80), xlabel='lookback time [Gyr]', ylabel=r'$\psi$ [deg]')
r = a.twinx()
EPS = {'eps2': ('k', '-', r'DM quadrupole $\epsilon_2$'), 'eps_elong': ('C0', '--', r'in-plane part $\epsilon_{\rm elong}$')}   # tilt part: printed
for qn, (col, ls, lab) in EPS.items():
    e = by_epoch(T, qn)
    r.plot(e.index, e.m, ls, color=col, lw=1.2, label=lab)
for qn in ('eps2', 'eps_tilt', 'eps_elong'):
    e = by_epoch(T, qn)
    print(f'(c) {qn}: ' + '; '.join(f'{t:.1f}: {v:.4f}' for t, v in e.m.items()))
r.set(yscale='log', ylim=(1e-3, 0.04))
r.set_ylabel(r'DM quadrupole $\epsilon_2$ at 2 kpc')
print('(c) psi: ' + '; '.join(f'{t:.1f}: {v:.0f}' for t, v in psi.m.items()))
h3 = [h for x in (a, r) for h in x.get_legend_handles_labels()[0]]
a.legend(h3, [h.get_label() for h in h3], loc='upper right', **LEG)

for a, title in zip(ax, ('(a) loss of rotation', '(b) role of halo shape', '(c) halo settling')):
    a.set_title(title, loc='left', fontsize=8)   # above the panel, as Fig. sims_sidm
ax[0].legend(loc='upper right', ncol=2, **LEG)
fig.savefig(ROOT / 'figures/paper_letter_scramble.png', dpi=250)
print('wrote figures/paper_letter_scramble.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_letter_scramble.png')

**Figure 4.** How the early TNG50 cohorts lose their rotation, and why. (a) $v_\phi/\sigma_z$ about each
cohort's spin at birth against time since birth (inner labels: orbital periods at 2 kpc) for the real stars
(black) and for test particles in the progenitor's gravity frozen at birth (orange), with only the dark matter aspherical (red) and
made axisymmetric about the cohort's spin (purple); blue band: the star-forming gas; dotted: the LMC's [Fe/H] <
-1.5 giants (at $R$ = 4 kpc, for reference). (b) $v_\phi/\sigma_z$ after
0.2–0.35 Gyr against the dark-matter quadrupole $\epsilon_2$ at birth; dashed: at birth. (c)
Misalignment $\psi$ of the halo's minor axis with the disc spin (red, left), and $\epsilon_2$ (black) with its in-plane part (blue
dashed; right), against lookback time. The rotation is halved within one orbital period, and mostly lost within two, because the
halo is aspherical and misaligned with the disc; it settles only after the LMC's oldest stars formed.

## How the inputs are made

The figure script reads the cohorts and their test-particle runs (`results/sims_scramble.csv`), the star-forming gas
(`results/sims_heating_gas.csv`), the halo shapes (`results/sims_halo_settle.csv`), the orbital periods
(`results/sims_vcirc.csv`) and the LMC fits (`results/kin_support_ellipsoid.csv`). The logged steps upstream of the figure,
nearest first:

In [ ]:
import kbtools as kb
kb.upstream('paper_letter_scramble.png')

The test-particle experiment (`scripts/sims_scramble.py`, a few minutes) and the halo shapes (`scripts/sims_halo_settle.py`) use
AGAMA multipole potentials built from the TNG50 progenitors (`scripts/fetch_tng50_progenitors.py`). For an introduction to
potentials and orbit integration with AGAMA, see the
[Galactic Dynamics with AGAMA](https://vasilybelokurov.github.io/galactic-dynamics-tutorial/) tutorial. The same experiment on the
Auriga analogues: `scripts/auriga_scramble.py`.

## Try it: the experiment for one cohort

The test-particle experiment needs the TNG50 particles, which are too large for this site. One cohort can be repeated here from a
few small files: the stars formed in the 25 Myr before snapshot 33 in progenitor 670104 (the $z$ = 0 subhalo id in
TNG50-1). Its real stars lose their rotation within 0.5 Gyr, like most early cohorts in panel (a) of
Figure 4.

The gravity of the progenitor is taken at the cohort's birth snapshot and held fixed. It is used in two forms: complete (all the dark
matter, gas and stars), and with every part that is not axisymmetric about the cohort's spin removed, separately for the dark matter
and for the baryons. The stars are then moved as test particles through each form for 0.5 Gyr, and their
$v_\phi/\sigma_z$ about the cohort's spin at birth is measured. If the non-axisymmetric gravity removes the rotation, the test
particles lose it in the complete gravity and keep it in the axisymmetric one.

### The inputs for one cohort

The step below, run on the full simulation, picks the cohort and its stars as `scripts/sims_scramble.py` does and writes what the two
runs need: the AGAMA multipole coefficients of the two forms of the gravity, the stars' positions, velocities and masses at birth in
the frame aligned with the cohort's spin, and the times of the later snapshots.

<details><summary>About <code>scripts/build_scramble_demo.py</code></summary>

```text
Inputs of the runnable one-cohort scrambling experiment of the knowledge base (KB page of Letter Fig. 4, scripts/scramble_demo.py).

scripts/sims_scramble.py moves the stars of every early TNG50 cohort as test particles through AGAMA potentials of their progenitor
and needs the TNG50 particle cutouts. This script extracts, for one cohort (COHORT: progenitor 670104, born at snapshot 33,
z = 2.0), the few things that the frozen-at-birth variants E and D0_fr of that experiment need, so that scripts/scramble_demo.py can
repeat them without the cutouts. It makes them with the functions of scripts/sims_scramble.py and the same steps as its run_cohort
(the cohort, the integration frame, the expansions at the birth snapshot); these steps are restated here, not changed.

Reads: the TNG50 full cutouts of the progenitor at snapshots 21-40 and its SubLink tree (lmcold.sims.progenitor, main_branch;
scripts/fetch_tng50_progenitors.py), in ~/data.
Writes (integration frame of sims_scramble.py: origin at the stellar centre, z along the cohort's spin at its birth snapshot;
kpc, km/s, Msun):
  results/scramble_demo_E.ini                 AGAMA Multipole coefficients of all particles (dark matter, gas, stars) within 60 kpc
                                              at the birth snapshot: variant E (all gravity, frozen at birth);
  results/scramble_demo_D0_fr_dm.ini,         the dark matter and the baryons (gas + stars) in separate expansions with only the
  results/scramble_demo_D0_fr_bar.ini         m = 0 terms kept: variant D0_fr (all gravity axisymmetric about the cohort's spin);
  results/scramble_demo_stars.csv             the cohort members followed in v_phi/sigma_z (x, y, z, vx, vy, vz, m);
  results/scramble_demo_centre.csv            the 500 stars within 2 kpc whose mean force defines the frame's acceleration (x, y, z);
  results/scramble_demo_times.csv             the times of the snapshots after birth at which v_phi/sigma_z is measured (t in
                                              AGAMA's unit 1 kpc / (km/s) = 0.9778 Gyr, t_gyr in Gyr).
Run: .venv/bin/python scripts/manifest.py run build_scramble_demo.py   (needs the TNG50 cutouts and AGAMA)
```
</details>

*Shown, not run here: the TNG50 particle cutouts (scripts/fetch_tng50_progenitors.py); a few seconds. The cells below are the script itself; run it from the repository with* `python scripts/build_scramble_demo.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The cohort, named by its progenitor (z = 0 subhalo id of TNG50-1) and its birth snapshot, and the functions of the full experiment.

```python
import shutil
import tempfile

import numpy as np
import pandas as pd

from fetch_tng50_progenitors import FULL_SNAPS
from lmcold import sims
from lmcold.sources import ROOT
from sims_scramble import (DTMAX, KPC_KMS_GYR, M0, R_COHORT, R_MEMBER, WINDOW, expansion, frame, in_ring, keep_terms, load,
                           particles)

COHORT = ('670104', 33)   # progenitor, birth snapshot: its real stars and variant E lose their rotation, variant D0_fr keeps it
OUT = ROOT / 'results'    # output directory of the demo inputs
FMT = '%.17g'             # full float64 precision, so that the demo starts from the same numbers as the full run
```

#### Load the data
The progenitor's particles at every snapshot 21-40 (z = 4-1.5), centred on its stars as in the full run (`sims_scramble.load`).

```python
name, s_birth = COHORT
# SubLink main branch of the progenitor: its subhalo id at each snapshot.
branch = sims.main_branch(name)
# One dict per snapshot with a cutout (keys used below: 's' snapshot, 'lb' lookback time [Gyr], 'a' scale factor, 'st' stars).
snaps = [d for d in (load(name, s, branch) for s in FULL_SNAPS) if d is not None]
# Index and data of the birth snapshot.
i0 = next(i for i, c in enumerate(snaps) if c['s'] == s_birth)
c0 = snaps[i0]
```

#### The cohort and its frame
As in `sims_scramble.one` and `run_cohort`: the stars formed in the 25 Myr before the birth snapshot within 3 kpc; their members
within 10 kpc at birth; the subset at r < 3 kpc and cylindrical R > 0.5 kpc whose $v_\phi/\sigma_z$ is followed (`in_ring`); the
frame with z along the members' spin; the snapshots up to 0.5 Gyr after the mean formation time.

```python
st = c0['st']
age = st['age_form'] - c0['lb']                                                    # Gyr formed before the birth snapshot
k = (age >= 0) & (age < WINDOW) & (np.linalg.norm(st['x'], axis=1) < R_COHORT)     # the cohort
t_form = float(np.average(st['age_form'][k], weights=st['m'][k]))                  # mean formation time (lookback, Gyr)
w0 = np.isin(st['ids'], st['ids'][k]) & (np.linalg.norm(st['x'], axis=1) < R_MEMBER)
xs, vs, ms = st['x'][w0], st['v'][w0], st['m'][w0]                                 # members at birth (kpc, km/s, Msun)
sel = in_ring(xs, vs, ms)                                                          # followed in v_phi/sigma_z
n = (ms[:, None] * np.cross(xs, vs)).sum(0)
n /= np.linalg.norm(n)                                                             # cohort's spin at birth
R = frame(n)                                                                       # rows: integration-frame axes
later = [i for i in range(i0, len(snaps)) if t_form - snaps[i]['lb'] <= DTMAX + 1e-6]
t = np.array([(c0['lb'] - snaps[i]['lb']) / KPC_KMS_GYR for i in later])           # AGAMA time units since birth
xc = st['x'][np.linalg.norm(st['x'], axis=1) < 2] @ R.T                            # stars within 2 kpc, integration frame
cen = xc[np.random.default_rng(0).choice(len(xc), min(500, len(xc)), replace=False)]   # as run_cohort's pick (seed 0)
print(f'{name} snapshot {s_birth}: z = {1 / c0["a"] - 1:.2f}, {len(ms)} members, {sel.sum()} followed, {len(t)} times')
```

#### Write the inputs
The expansions at the birth snapshot (all particles; dark matter and baryons separately with only m = 0 kept), the followed
stars in the integration frame, the frame-acceleration stars and the times.

```python
# Variant E: one Multipole expansion of all particles (integration frame R), written to a temporary file and copied to results/.
with tempfile.TemporaryDirectory() as tmp:
    shutil.copy(expansion(*particles(c0, ('dm', 'gas', 'st'), R), f'{tmp}/all.ini'), OUT / 'scramble_demo_E.ini')
    # Variant D0_fr: dark matter and baryons expanded separately, then only the m = 0 terms kept (M0, sims_scramble.keep_terms).
    for part, parts in (('dm', ('dm',)), ('bar', ('gas', 'st'))):
        f = expansion(*particles(c0, parts, R), f'{tmp}/{part}.ini')
        shutil.copy(keep_terms(f, M0, 'D0_fr'), OUT / f'scramble_demo_D0_fr_{part}.ini')
# Followed members rotated into the integration frame: positions [kpc] and velocities [km/s], shape (N, 3) each.
x, v = xs[sel] @ R.T, vs[sel] @ R.T
pd.DataFrame(np.column_stack([x, v, ms[sel]]), columns=['x', 'y', 'z', 'vx', 'vy', 'vz', 'm']).to_csv(
    OUT / 'scramble_demo_stars.csv', index=False, float_format=FMT)
# Frame-acceleration stars [kpc] and the measurement times (AGAMA units and Gyr).
pd.DataFrame(cen, columns=['x', 'y', 'z']).to_csv(OUT / 'scramble_demo_centre.csv', index=False, float_format=FMT)
pd.DataFrame({'t': t, 't_gyr': t * KPC_KMS_GYR}).to_csv(OUT / 'scramble_demo_times.csv', index=False, float_format=FMT)
print('wrote results/scramble_demo_*')
```

### The two runs

The orbit integration needs AGAMA. On this site it is installed; in Google Colab the next cell installs a ready-made copy (the
wheel of the [Galactic Dynamics with AGAMA](https://vasilybelokurov.github.io/galactic-dynamics-tutorial/) tutorial).

In [ ]:
# Install AGAMA only if it is missing (e.g. in Colab): a ready-made Linux wheel for this Python version.
import importlib.util, subprocess, sys
if importlib.util.find_spec('agama') is None:
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'https://github.com/vasilybelokurov/galactic-dynamics-tutorial/'
                    f'releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl'], check=True)

The script below runs the two forms of the gravity, compares the result with the full experiment (`results/sims_scramble.csv`) and
stops if they disagree; Figure 4b shows the result.

<details><summary>About <code>scripts/scramble_demo.py</code></summary>

```text
The scrambling experiment of Letter Fig. 4 for one TNG50 cohort, runnable without the simulation (KB page of Letter Fig. 4).

The stars of the cohort (progenitor 670104, born at snapshot 33, z = 2.0; results/scramble_demo_*, scripts/build_scramble_demo.py)
are moved as test particles for 0.5 Gyr through two AGAMA potentials of their progenitor's gravity at the birth snapshot, held
fixed (frozen at birth), as the variants E and D0_fr of scripts/sims_scramble.py:
  E      all the gravity (dark matter, gas and stars in one Multipole expansion, lmax 6);
  D0_fr  the dark matter and the baryons in two expansions with only the m = 0 terms kept: axisymmetric about the cohort's spin.
The frame is the full run's: origin at the stellar centre, z along the cohort's spin at birth. Because the origin follows the stars,
each potential gets the full run's UniformAcceleration term, minus the mean force on 500 stars within 2 kpc (constant for a frozen
potential). At the snapshot times after birth, v_phi/sigma_z of the stars about the z axis (sims_scramble.support: mass-weighted
mean azimuthal velocity over the vertical velocity dispersion) is compared with vs_sim_birth of results/sims_scramble.csv; the
script stops if any value differs by more than TOL.
Reads results/scramble_demo_*.{ini,csv} and results/sims_scramble.csv. Writes results/scramble_demo.csv (variant, t [Gyr since the
birth snapshot], vs: v_phi/sigma_z) and figures/scramble_demo.png.
Run: .venv/bin/python scripts/manifest.py run scramble_demo.py   (needs AGAMA; a few seconds)
```
</details>

**Data read by `scripts/scramble_demo.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/scramble_demo_centre.csv` | repository | written by `python scripts/build_scramble_demo.py` |
| `results/scramble_demo_stars.csv` | repository | written by `python scripts/build_scramble_demo.py` |
| `results/scramble_demo_times.csv` | repository | written by `python scripts/build_scramble_demo.py` |
| `results/sims_scramble.csv` | repository | written by `python scripts/sims_scramble.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The cohort, the tolerance of the comparison with the full run, and the functions of the full experiment
(`scripts/sims_scramble.py`; importing it needs AGAMA but no simulation files).

In [ ]:
import matplotlib

matplotlib.use('Agg')
import agama
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold.layout import panel_grid
from lmcold.sources import ROOT
from sims_scramble import KPC_KMS_GYR, accel, support

COHORT = ('670104', 33)   # progenitor (z = 0 subhalo id), birth snapshot: as scripts/build_scramble_demo.py
N_GRID = 101             # times of the curves in the figure, 0-0.46 Gyr
TOL = 0.01                # largest allowed |difference| of v_phi/sigma_z from the full run (its CSV is rounded to 5 decimals)
RES = ROOT / 'results'

#### Load the data
The cohort's stars in the integration frame (positions in kpc, velocities in km/s, masses in Msun), the 500 stars that define the
frame's acceleration, the snapshot times after birth, the coefficient files of the two potentials, and the full run's values.

In [ ]:
stars = pd.read_csv(RES / 'scramble_demo_stars.csv')          # the stars followed in v_phi/sigma_z (r < 3, R > 0.5 kpc at birth)
cen = pd.read_csv(RES / 'scramble_demo_centre.csv').values    # (500, 3) kpc: stars within 2 kpc at birth
times = pd.read_csv(RES / 'scramble_demo_times.csv')          # t: AGAMA time unit (1 kpc / (km/s) = 0.9778 Gyr); t_gyr: Gyr
p_all = agama.Potential(str(RES / 'scramble_demo_E.ini'))             # all particles, all terms
p_dm = agama.Potential(str(RES / 'scramble_demo_D0_fr_dm.ini'))       # dark matter, m = 0 terms
p_bar = agama.Potential(str(RES / 'scramble_demo_D0_fr_bar.ini'))     # gas + stars, m = 0 terms
full = pd.read_csv(RES / 'sims_scramble.csv', dtype={'halo': str}).query('halo == @COHORT[0] and s_birth == @COHORT[1]')

#### The two potentials
Each frozen potential plus the frame's acceleration: AGAMA's UniformAcceleration term with minus the mean force on the 500 stars,
given at the first and last times (the same value: the potential does not change), as in `sims_scramble.run_cohort`.

In [ ]:
t = times.t.values
ic = stars[['x', 'y', 'z', 'vx', 'vy', 'vz']].values          # (N, 6) initial conditions at the birth snapshot
m = stars.m.values
p_axi = agama.Potential(p_dm, p_bar)
pots = {'E': agama.Potential(p_all, accel([p_all, p_all], [cen, cen], t[[0, -1]])),
        'D0_fr': agama.Potential(p_dm, p_bar, accel([p_axi, p_axi], [cen, cen], t[[0, -1]]))}

#### Orbits and rotational support
The orbits are integrated to the last time and read from AGAMA's interpolated trajectory, as in the full run: at the snapshot
times (for the comparison) and on a grid of N_GRID times (for the curves); $v_\phi/\sigma_z$ about the cohort's spin at birth (the
z axis) at each time.

In [ ]:
tt = np.union1d(np.linspace(0, t[-1], N_GRID), t)              # AGAMA time units: the grid and the snapshot times
rows = []
for var, pot in pots.items():
    orbs = agama.orbit(potential=pot, ic=ic, time=t[-1], dtype=object, verbose=False)
    w6 = np.stack([o(tt) for o in orbs], axis=1)              # (time, star, 6): kpc, km/s
    rows += [dict(variant=var, t=tk * KPC_KMS_GYR, vs=support(s[:, :3], s[:, 3:], m, np.array([0., 0., 1.]))) for tk, s in zip(tt, w6)]
demo = pd.DataFrame(rows)
demo.to_csv(RES / 'scramble_demo.csv', index=False, float_format='%.5f')

#### Comparison with the full run
The same quantity in `results/sims_scramble.csv` (`vs_sim_birth`), matched by variant and time.

In [ ]:
key = lambda d: d.assign(it=(d.t * 1e5).round().astype(int))   # time in units of 1e-5 Gyr: the full run's rounding
cmp = key(demo).merge(key(full)[['variant', 'it', 'vs_sim_birth']], on=['variant', 'it'])   # the snapshot times only
dev = (cmp.vs - cmp.vs_sim_birth).abs()
print(cmp.assign(diff=cmp.vs - cmp.vs_sim_birth)[['variant', 't', 'vs', 'vs_sim_birth', 'diff']].to_string(index=False))
print(f'largest |difference| from the full run: {dev.max():.2e} (tolerance {TOL})')
assert len(cmp) == 2 * len(t) and dev.max() < TOL, 'the demo does not reproduce the full run'

#### Figure
$v_\phi/\sigma_z$ against time since birth: the test particles of this run (lines), and at the snapshot times those of the full
run (open circles) and the real stars (black squares). The figure is saved at the end of this cell.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, a = panel_grid(1, 1, width=3.4, top=0.75, aspect=0.75)   # room above the panel for the legend
a = a[0, 0]
VAR = {'E': ('all gravity, frozen at birth (E)', 'C1'), 'D0_fr': ('axisymmetric about the spin (D0_fr)', 'C4')}
for var, (lab, col) in VAR.items():
    d, f = demo[demo.variant == var], full[full.variant == var]
    a.plot(d.t, d.vs, '-', color=col, lw=1.2, label=lab)
    a.plot(f.t, f.vs_sim_birth, 'o', color=col, mfc='none', ms=5)
real = full[full.variant == 'E']                               # the real stars' values are repeated in every variant's rows
a.plot(real.t, real.vs_meas_birth, 'ks--', ms=3, lw=0.8, label='real stars (TNG50)')
a.plot([], [], 'o', color='0.4', mfc='none', ms=5, label='full run (sims_scramble.csv)')   # legend entry of the open circles
a.axhline(0, color='0.6', lw=0.5)
a.set(xlabel='time since birth [Gyr]', ylabel=r'$v_\phi/\sigma_z$', xlim=(-0.02, 0.5), ylim=(-2.2, 3.2))
a.legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=2, fontsize=5, frameon=False)   # above the panel
fig.savefig(ROOT / 'figures/scramble_demo.png', dpi=250)
print('wrote figures/scramble_demo.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/scramble_demo.png')

**Figure 4b.** $v_\phi/\sigma_z$ about the cohort's spin at birth against time since birth, for the cohort born at
snapshot 33 in TNG50 progenitor 670104: test particles in the progenitor's gravity frozen at birth, complete (orange) and made
axisymmetric about the cohort's spin (purple), computed here (lines) and in the full experiment (open circles), and the real stars
(black squares). The test particles lose their rotation in the complete gravity, as the real stars do, and keep it in the
axisymmetric gravity.